In [ ]:
!apt-get update
!apt-get install -y nmap

!pip install streamlit plotly pandas requests python-nmap -q

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,931 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,309 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,946 kB]
Get:14 http://archive.ubuntu

In [ ]:
TARGETS = [
    "testasp.vulnweb.com",
    "testphp.vulnweb.com",
    "zero.webappsecurity.com"
]

SCAN_OUTPUT_DIR = "scan_xml"

In [ ]:
import os
import subprocess

os.makedirs(SCAN_OUTPUT_DIR, exist_ok=True)

for target in TARGETS:

    output_file = f"{SCAN_OUTPUT_DIR}/{target}.xml"

    cmd = [
        "nmap",
        "-sV",
        "-oX",
        output_file,
        target
    ]

    subprocess.run(cmd)

print("Scanning complete")

Scanning complete


In [ ]:
import pandas as pd
import xml.etree.ElementTree as ET

results = []

for target in TARGETS:

    xml_file = f"{SCAN_OUTPUT_DIR}/{target}.xml"

    tree = ET.parse(xml_file)
    root = tree.getroot()

    for host in root.findall("host"):

        ip = host.find("address").get("addr")

        ports = host.find("ports")

        if ports is None:
            continue

        for port in ports.findall("port"):

            port_id = port.get("portid")

            service = port.find("service")

            if service is not None:
                service_name = service.get("name")
            else:
                service_name = "unknown"

            results.append({
                "ip": ip,
                "port": port_id,
                "service": service_name
            })

df = pd.DataFrame(results)

df.head()

,ip,port,service
0,44.238.29.244,80,http
1,54.82.22.214,80,http
2,54.82.22.214,443,https
3,54.82.22.214,8080,http


In [ ]:
import requests

VT_API_KEY = "YOUR_VT_API_KEY"
SHADON_API_KEY = "YOUR_SHADON_API_KEY"

In [ ]:
SENDER_EMAIL = "sender@gmail.com"
APP_PASSWORD = "YOUR_GMAIL_APP_PASSWORD"
RECEIVER_EMAIL = "receiver@gmail.com"

In [ ]:
import sys
!{sys.executable} -m pip install fpdf

In [ ]:
def check_virustotal(ip):

    url = f"https://www.virustotal.com/api/v3/ip_addresses/{ip}"

    headers = {
        "x-apikey": VT_API_KEY
    }

    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        return "Unknown"

    data = response.json()

    malicious = data["data"]["attributes"]["last_analysis_stats"]["malicious"]

    if malicious > 5:
        return "High"
    elif malicious > 0:
        return "Medium"
    else:
        return "Low"

In [ ]:
HIGH_RISK_SERVICES = {
    "ftp":3,
    "telnet":4,
    "ssh":1
}

def calculate_risk(row):

    score = 1

    if row["service"] in HIGH_RISK_SERVICES:
        score += HIGH_RISK_SERVICES[row["service"]]

    # Incorporate Shodan vulnerability count into risk score
    if "shodan_vuln_count" in row and row["shodan_vuln_count"] > 0:
        score += row["shodan_vuln_count"] # Each Shodan reported vuln adds to the risk

    # Add penalty based on VirusTotal reputation
    if row["vt_reputation"] == "Medium":
        score += 2
    elif row["vt_reputation"] == "High":
        score += 4

    return score

In [ ]:
df["vt_reputation"] = df["ip"].apply(check_virustotal)
df["risk_score"] = df.apply(calculate_risk, axis=1)

In [ ]:
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from datetime import datetime

def send_email_alert(df,target):

    high_risk = df[df["severity"].isin(["High","Critical"])]

    if high_risk.empty:
        print("No high risk vulnerabilities — email not sent")
        return

    msg = MIMEMultipart()

    msg["From"] = SENDER_EMAIL
    msg["To"] = RECEIVER_EMAIL
    msg["Subject"] = f"[SECURITY ALERT] High risk vulnerabilities detected on {target}"

    html = f"""
    <h2>CyberScan Pro Alert</h2>

    Target: {target}<br>
    Scan Time: {datetime.now()}<br>

    <h3>High Risk Findings</h3>

    {high_risk.to_html(index=False)}

    <hr>
    <i>Automatically generated by CyberScan Pro.</i>
    """

    msg.attach(MIMEText(html,"html"))

    server = smtplib.SMTP("smtp.gmail.com",587)
    server.starttls()
    server.login(SENDER_EMAIL,APP_PASSWORD)
    server.sendmail(SENDER_EMAIL,RECEIVER_EMAIL,msg.as_string())
    server.quit()

    print("Alert email sent")

In [ ]:
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from datetime import datetime

# Define a function to assign severity based on risk_score and vt_reputation
def assign_severity(row):
    if row["vt_reputation"] == "High" or row["risk_score"] >= 4:
        return "High"
    elif row["vt_reputation"] == "Medium" or row["risk_score"] >= 2:
        return "Medium"
    else:
        return "Low"

# Apply the function to create the 'severity' column
df["severity"] = df.apply(assign_severity, axis=1)

df.to_csv("scan_results.csv", index=False)
send_email_alert(df, "testphp.vulnweb.com")
print("scan_results.csv saved")

No high risk vulnerabilities — email not sent
scan_results.csv saved


In [ ]:
%%writefile dashboard.py

import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os
import subprocess
import xml.etree.ElementTree as ET
import requests
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from datetime import datetime
import nmap # Import the nmap library
from fpdf import FPDF # Added FPDF import

# =============================
# Global Variables and Configuration
# =============================

CONFIG = {
    "TARGETS": [
        "testasp.vulnweb.com",
        "testphp.vulnweb.com",
        "zero.webappsecurity.com"
    ],
    "SCAN_OUTPUT_DIR": "scan_xml",
    "VT_API_KEY": "YOUR_VT_API_KEY", # Your actual VirusTotal API Key
    "SHODAN_API_KEY": "YOUR_SHADON_API_KEY", # Your Shodan API Key
    "SENDER_EMAIL": "sender@gmail.com", # Your sender email
    "APP_PASSWORD": "YOUR_GMAIL_APP_PASSWORD", # Your app password
    "RECEIVER_EMAIL": "receiver@gmail.com", # Your receiver email
    "HIGH_RISK_SERVICES": {
        "ftp":3,
        "telnet":4,
        "ssh":1
    }
}


def check_virustotal(ip):

    url = f"https://www.virustotal.com/api/v3/ip_addresses/{ip}"

    headers = {
        "x-apikey": CONFIG["VT_API_KEY"]
    }

    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        return "Unknown"

    data = response.json()

    malicious = data["data"]["attributes"]["last_analysis_stats"]["malicious"]

    if malicious > 5:
        return "High"
    elif malicious > 0:
        return "Medium"
    else:
        return "Low"

def check_shodan(ip):
    if not CONFIG["SHODAN_API_KEY"] or CONFIG["SHODAN_API_KEY"] == "YOUR_SHODAN_API_KEY":
        return {"shodan_country": "N/A", "shodan_vuln_count": 0, "shodan_ports": "N/A"}

    url = f"https://api.shodan.io/shodan/host/{ip}?key={CONFIG["SHODAN_API_KEY"]}"
    try:
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            data = response.json()
            country = data.get("country_name", "Unknown")
            # Shodan often lists 'vulns' as a dictionary under data.get('vulns', {}) or simply 'vulns' as a list of strings
            # For simplicity, we'll count entries in a 'vulns' key if it exists and is a list/dict.
            vuln_count = len(data.get("vulns", [])) if isinstance(data.get("vulns"), list) else 0
            if isinstance(data.get("vulns"), dict):
                vuln_count = len(data.get("vulns").keys()) # If vulns is a dict of vulnerabilities

            ports = data.get("ports",[])
            shodan_ports = ", ".join(map(str, ports)) if ports else "N/A"

            return {"shodan_country": country, "shodan_vuln_count": vuln_count, "shodan_ports": shodan_ports}
        else:
            return {"shodan_country": "Unknown", "shodan_vuln_count": 0, "shodan_ports": "N/A"}
    except requests.exceptions.RequestException:
        return {"shodan_country": "Error", "shodan_vuln_count": 0, "shodan_ports": "N/A"}

def calculate_risk(row):

    score = 1

    if row["service"] in CONFIG["HIGH_RISK_SERVICES"]:
        score += CONFIG["HIGH_RISK_SERVICES"][row["service"]]

    # Incorporate Shodan vulnerability count into risk score
    if "shodan_vuln_count" in row and row["shodan_vuln_count"] > 0:
        score += row["shodan_vuln_count"] # Each Shodan reported vuln adds to the risk

    # Add penalty based on VirusTotal reputation
    if row["vt_reputation"] == "Medium":
        score += 2
    elif row["vt_reputation"] == "High":
        score += 4

    return score

def assign_severity(row):
    # Simplify the function to rely solely on risk_score thresholds (CVSS-like)
    if row["risk_score"] >= 8:
        return "Critical"
    elif row["risk_score"] >= 5:
        return "High"
    elif row["risk_score"] >= 2:
        return "Medium"
    else:
        return "Low"

def send_email_alert(df,target):

    # Reverted to include only 'High' and 'Critical' severity for alerts
    high_risk = df[df["severity"].isin(["High","Critical"])]

    if high_risk.empty:
        return "No high risk vulnerabilities — email not sent"

    msg = MIMEMultipart()

    msg["From"] = CONFIG["SENDER_EMAIL"]
    msg["To"] = CONFIG["RECEIVER_EMAIL"]
    msg["Subject"] = f"[SECURITY ALERT] High risk vulnerabilities detected on {target}"

    html = f"""
    <h2>CyberScan Pro Alert</h2>

    Target: {target}<br>
    Scan Time: {datetime.now()}<br>

    <h3>High Risk Findings</h3>

    {high_risk.to_html(index=False)}

    <hr>
    <i>Automatically generated by CyberScan Pro.</i>
    """

    msg.attach(MIMEText(html,"html"))

    try:
        server = smtplib.SMTP("smtp.gmail.com",587)
        server.starttls()
        server.login(CONFIG["SENDER_EMAIL"],CONFIG["APP_PASSWORD"])
        server.sendmail(CONFIG["SENDER_EMAIL"],CONFIG["RECEIVER_EMAIL"],msg.as_string())
        server.quit()
        return "Alert email sent successfully."
    except Exception as e:
        return f"Failed to send email alert: {e}"

def generate_pdf_report(df, target):
    high_risk_df = df[df["severity"].isin(["High", "Critical"])].copy()

    pdf = FPDF()
    pdf.add_page()

    pdf.set_font("Arial", "B", 16)
    pdf.cell(0, 10, "CyberScan Pro Vulnerability Report", 0, 1, "C")
    pdf.ln(5)

    pdf.set_font("Arial", "", 12)
    pdf.cell(0, 10, f"Target: {target}", 0, 1, "L")
    pdf.cell(0, 10, f"Scan Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", 0, 1, "L")
    pdf.ln(10)

    pdf.set_font("Arial", "B", 14)
    pdf.cell(0, 10, "High-Risk Findings", 0, 1, "L")
    pdf.ln(5)

    if not high_risk_df.empty:
        pdf.set_font("Arial", "", 10)
        # Prepare data for table
        headings = [col.replace('_', ' ').title() for col in high_risk_df.columns]
        col_widths = [20, 15, 25, 25, 25, 20, 20, 25, 25, 25, 20] # Default widths

        # Adjust column widths based on the actual columns present in high_risk_df
        actual_headings = high_risk_df.columns.tolist()
        adjusted_col_widths = []
        for h in headings:
            if h.replace(' ', '_').lower() in [col.lower() for col in actual_headings]:
                # Simple width allocation, could be improved for dynamic content
                if 'ip' in h.lower(): adjusted_col_widths.append(25)
                elif 'port' in h.lower(): adjusted_col_widths.append(15)
                elif 'service' in h.lower(): adjusted_col_widths.append(25)
                elif 'risk' in h.lower(): adjusted_col_widths.append(25)
                elif 'severity' in h.lower(): adjusted_col_widths.append(20)
                elif 'country' in h.lower(): adjusted_col_widths.append(25)
                elif 'vuln' in h.lower(): adjusted_col_widths.append(20)
                elif 'shodan' in h.lower(): adjusted_col_widths.append(20)
                else: adjusted_col_widths.append(25) # Default for others

        # Ensure we don't have more widths than columns or vice versa
        if len(adjusted_col_widths) != len(headings):
            # Fallback to a simpler, equal distribution if dynamic fails
            col_width = pdf.w / (len(headings) + 1)
            adjusted_col_widths = [col_width] * len(headings)

        # Add table header
        for i, header in enumerate(headings):
            pdf.cell(adjusted_col_widths[i], 10, str(header), 1, 0, 'C')
        pdf.ln()

        # Add table rows
        for row in high_risk_df.values.tolist():
            for i, item in enumerate(row):
                pdf.cell(adjusted_col_widths[i], 10, str(item), 1, 0, 'L')
            pdf.ln()
    else:
        pdf.set_font("Arial", "I", 12)
        pdf.cell(0, 10, "No high-risk vulnerabilities found.", 0, 1, "L")

    output_filename = f"scan_report_{target.replace('.', '_').replace(' ', '_')}_{datetime.now().strftime('%Y%m%d%H%M%S')}.pdf"
    pdf.output(output_filename)
    return output_filename

# Helper functions for run_full_scan_logic
def _initialize_scan_environment(output_dir, status_container):
    status_container.update(label="Initializing scan environment... (0%)", state="running")
    os.makedirs(output_dir, exist_ok=True)
    nm = nmap.PortScanner()
    status_container.write("Scan environment initialized.")
    return nm

def _perform_nmap_scan(nm, targets, output_dir, status_container):
    status_container.update(label="Running Nmap scan... (10%)", state="running")
    for i, target in enumerate(targets):
        output_file = f"{output_dir}/{target}.xml"
        try:
            # Removed -oX argument, now just perform the scan
            nm.scan(target, arguments='-sV -Pn')
            # Retrieve the XML output after the scan and decode it
            with open(output_file, 'w') as f:
                f.write(nm.get_nmap_last_output().decode('utf-8')) # Decode and write
        except nmap.PortScannerError as e:
            status_container.error(f"Nmap scan failed for {target}: {e}")
            continue
        progress_percentage = 10 + int((i+1)/len(targets) * 30)
        status_container.update(label=f"Scanning {target} with Nmap... ({progress_percentage}%)", state="running")
    status_container.write("Nmap scanning complete.")
    return nm

def _parse_nmap_results(nm, status_container):
    status_container.update(label="Parsing scan results... (40%)", state="running")
    results = []
    for host in nm.all_hosts():
        for proto in nm[host].all_protocols():
            lport = nm[host][proto].keys()
            for port_id in lport:
                port_details = nm[host][proto][port_id]
                service_name = port_details.get('name', 'unknown')
                product = port_details.get('product', 'N/A')
                version = port_details.get('version', 'N/A')
                state = port_details.get('state', 'N/A')

                results.append({
                    "ip": host,
                    "port": port_id,
                    "service": service_name,
                    "product": product,
                    "version": version,
                    "state": state
                })
    df_scan = pd.DataFrame(results)
    status_container.write("Scan results parsed.")
    return df_scan

def _integrate_shodan_data(df_scan, shodan_api_key, status_container):
    if df_scan.empty:
        return df_scan
    status_container.update(label="Checking Shodan for additional intelligence... (70%)", state="running")
    unique_ips = df_scan["ip"].unique()
    shodan_results = []
    for i, ip in enumerate(unique_ips):
        shodan_data = check_shodan(ip) # check_shodan already uses CONFIG
        shodan_results.append({"ip": ip, **shodan_data})
        progress_percentage = 70 + int((i+1)/len(unique_ips) * 10)
        status_container.update(label=f"Checking Shodan for {ip}... ({progress_percentage}%)", state="running")

    df_shodan = pd.DataFrame(shodan_results)
    df_scan = pd.merge(df_scan, df_shodan, on="ip", how="left").fillna({"shodan_country": "N/A", "shodan_vuln_count": 0, "shodan_ports": "N/A"})
    status_container.write("Shodan data integrated.")
    return df_scan

def _calculate_risks_and_severities(df_scan, status_container):
    if df_scan.empty:
        return df_scan
    status_container.update(label="Calculating risks and severities... (80%)", state="running")
    df_scan["vt_reputation"] = df_scan["ip"].apply(check_virustotal) # check_virustotal already uses CONFIG
    df_scan["risk_score"] = df_scan.apply(calculate_risk, axis=1) # calculate_risk already uses CONFIG
    df_scan["severity"] = df_scan.apply(assign_severity, axis=1)
    status_container.write("Risks and severities calculated.")
    return df_scan

def _inject_mock_vulnerability(df_scan, status_container):
    # Always inject a mock high-risk vulnerability to ensure an alert is triggered for demonstration
    mock_high_vuln = pd.DataFrame({
        "ip": ["1.1.1.1"],
        "port": [1337],
        "service": ["mock_service"],
        "product": ["mock_product"],
        "version": ["mock_version"],
        "state": ["open"],
        "vt_reputation": ["High"],
        "risk_score": [10], # Set risk score to >= 8 to make it Critical
        "shodan_country": ["Mockland"],
        "shodan_vuln_count": [5],
        "shodan_ports": ["1337"],
        "severity": ["Critical"]
    })
    df_scan = pd.concat([df_scan, mock_high_vuln], ignore_index=True)
    status_container.info("Injected mock critical vulnerability for demonstration purposes.")
    return df_scan

def _save_results_and_alert(df_scan, target_name_for_email, status_container):
    if df_scan.empty: # Only proceed if df_scan is not empty
        status_container.warning("No scan data to save or alert on.")
        return

    df_scan.to_csv("scan_results.csv", index=False)
    status_container.success("Scan results processed and saved.")
    status_container.update(label="Sending email alerts... (95%)", state="running")

    high_risk_entries = df_scan[df_scan["severity"].isin(["High","Critical"])]
    if not high_risk_entries.empty:
        email_status_message = send_email_alert(df_scan, target_name_for_email)
        status_container.write(email_status_message)
    else:
        status_container.info("No high risk vulnerabilities found across all targets — email not sent")


def run_full_scan_logic():
    with st.status("Running full scan...", expanded=True) as status_container:
        nm = _initialize_scan_environment(CONFIG["SCAN_OUTPUT_DIR"], status_container)
        nm = _perform_nmap_scan(nm, CONFIG["TARGETS"], CONFIG["SCAN_OUTPUT_DIR"], status_container)
        df_scan = _parse_nmap_results(nm, status_container)

        if not df_scan.empty:
            df_scan = _integrate_shodan_data(df_scan, CONFIG["SHODAN_API_KEY"], status_container)
            df_scan = _calculate_risks_and_severities(df_scan, status_container)
            df_scan = _inject_mock_vulnerability(df_scan, status_container)
            _save_results_and_alert(df_scan, "Overall Scan Results", status_container)
        else:
            status_container.warning("No open ports or services found during scan.")

        status_container.update(label="Scan complete.", state="complete", expanded=False)
    st.cache_data.clear() # Clear cache to force reload data after scan

def load_scan_results():
    try:
        df = pd.read_csv("scan_results.csv")
        return df
    except FileNotFoundError:
        st.info("No scan run yet. Showing sample data. Please run a scan to get real results.")
        sample_df = pd.DataFrame({
            "ip":["192.168.1.1","192.168.1.1","192.168.1.2","192.168.1.2","192.168.1.3"],
            "port":[22,80,21,443,23],
            "service":["ssh","http","ftp","https","telnet"],
            "product":["OpenSSH","Apache","vsftpd","nginx","telnetd"],
            "version":["7.6p1","2.4.29","3.0.3","1.14.0","0.17"],
            "state":["open","open","open","open","open"],
            "vt_reputation":["Low","Low","Medium","Medium","High"],
            "risk_score":[1,3,6,7,10], # Adjusted risk_scores to fit new severity thresholds
            "shodan_country":["USA","USA","Germany","Germany","Canada"],
            "shodan_vuln_count":[0,0,1,1,3],
            "shodan_ports":["22,80","22,80","21,443","21,443","23"]
        })
        # Assign severity based on new risk_score thresholds for sample data
        sample_df["severity"] = sample_df.apply(assign_severity, axis=1)
        return sample_df

# =============================
# Streamlit App Layout
# =============================

st.set_page_config(
    page_title="CyberScan Pro",
    layout="wide"
)

st.title("\U0001F6A8 CyberScan Pro: Network Security Dashboard")
st.markdown("A powerful tool for network reconnaissance, threat intelligence, and vulnerability assessment.")
st.divider()

# Sidebar
st.sidebar.title("⚙️ Controls & Status")
st.sidebar.header("Application Status")

if CONFIG["VT_API_KEY"] and CONFIG["VT_API_KEY"] != "YOUR_VIRUSTOTAL_API_KEY":
    st.sidebar.success("\u2705 VirusTotal API Key configured")
else:
    st.sidebar.error("\u274C VirusTotal API Key NOT configured")

if CONFIG["SHODAN_API_KEY"] and CONFIG["SHODAN_API_KEY"] != "YOUR_SHODAN_API_KEY":
    st.sidebar.success("\u2705 Shodan API Key configured")
else:
    st.sidebar.error("\u274C Shodan API Key NOT configured")

if CONFIG["SENDER_EMAIL"] and CONFIG["APP_PASSWORD"] and CONFIG["RECEIVER_EMAIL"]:
    st.sidebar.success("\u2709\ufe0f Email alert system configured")
else:
    st.sidebar.warning("\u26A0\ufe0f Email alert system NOT fully configured")

with st.sidebar.expander("\U0001F3AF Current Scan Targets", expanded=False):
    for target in CONFIG["TARGETS"]:
        st.write(f"- {target}")
st.sidebar.divider()

st.sidebar.header("\U0001F680 Scan Actions")

if st.sidebar.button("\U0001F680 Run New Full Scan", help="Execute a new Nmap scan and process results."):
    run_full_scan_logic()

if st.sidebar.button("\U0001F504 Refresh Dashboard Data", help="Reload data from the last scan without running a new one."):
    st.cache_data.clear() # Clear cache to force reload data
    st.rerun()
st.sidebar.divider()

df = load_scan_results()

# --- Start: New logic for automatic email on dashboard refresh ---
if not df.empty:
    high_risk_entries_on_load = df[df["severity"].isin(["High", "Critical"])]
    if not high_risk_entries_on_load.empty:
        st.sidebar.info("High-risk entries detected on dashboard load. Attempting to send email alert...")
        email_status_refresh = send_email_alert(high_risk_entries_on_load, "CyberScan Pro - Dashboard Load Alert")
        st.sidebar.write(f"Automatic email alert status: {email_status_refresh}")
    else:
        st.sidebar.info("No high/critical risk entries detected on dashboard load; no automatic email sent.")
# --- End: New logic for automatic email on dashboard refresh ---

# Key Metrics
st.subheader("\U0001F4C8 Overview")
col1,col2,col3,col4,col5 = st.columns(5)

col1.metric("Total Hosts Found", df["ip"].nunique() if not df.empty else 0)
col2.metric("Open Ports Detected", len(df) if not df.empty else 0)
col3.metric("Unique Services", df["service"].nunique() if not df.empty else 0)
col4.metric("Highest Risk Score", df["risk_score"].max() if not df.empty else 0)
high_risk_count = len(df[df["severity"].isin(["High", "Critical"])]) if not df.empty else 0 # Include Critical
col5.metric("High Risk Entries", high_risk_count, delta=f"Up from last scan" if high_risk_count > 0 else None)

st.divider()

# Navigation Tabs
tab1,tab2,tab3,tab4 = st.tabs([
"\U0001F4C4 Scan Data",
"\U0001F4C8 Visualizations",
"\U0001F6A8 Threat Intelligence",
"\U0001F4BE Export Data"
])

with tab1:
    st.header("Detailed Scan Results")
    if not df.empty:
        col_filter1, col_filter2, col_filter3 = st.columns(3)
        with col_filter1:
            ip_filter = st.selectbox("Filter by IP Address", ["All"] + list(df["ip"].unique()), key="ip_filter")
        with col_filter2:
            service_filter = st.selectbox("Filter by Service Name", ["All"] + list(df["service"].unique()), key="service_filter")
        with col_filter3:
            shodan_countries = []
            if "shodan_country" in df.columns:
                shodan_countries = list(df["shodan_country"].unique())
            country_filter = st.selectbox("Filter by Shodan Country", ["All"] + shodan_countries, key="country_filter")

        filtered = df.copy()
        if ip_filter != "All":
            filtered = filtered[filtered["ip"]==ip_filter]
        if service_filter != "All":
            filtered = filtered[filtered["service"]==service_filter]
        if country_filter != "All":
            if "shodan_country" in filtered.columns: # Added check here as well
                filtered = filtered[filtered["shodan_country"]==country_filter]

        st.dataframe(filtered, use_container_width=True, hide_index=True)

        st.markdown("### \U0001F4CD Host Summary")
        host_summary_cols = {
            'open_ports': ("port","count"),
            'max_risk': ("risk_score","max"),
            'malicious_reputation_count': ('vt_reputation', lambda x: (x == 'High').sum() + (x == 'Medium').sum()),
            'services': ("service",lambda x:", ".join(x.unique())),
        }
        if "shodan_vuln_count" in df.columns:
            host_summary_cols["shodan_vulns"] = ('shodan_vuln_count', 'max')
        if "shodan_country" in df.columns:
            host_summary_cols["shodan_country"] = ("shodan_country", 'first')

        host_summary = df.groupby("ip").agg(**host_summary_cols).reset_index()
        st.dataframe(host_summary, use_container_width=True, hide_index=True)
    else:
        st.info("No scan data available. Run a full scan to populate results.")

with tab2:
    st.header("Data Visualizations")
    if not df.empty:
        col1,col2 = st.columns(2)
        with col1:
            ports_chart = px.bar(
                df.groupby("ip")["port"].count().reset_index(),
                x="ip",
                y="port",
                title="<b>Open Ports per Host</b>",
                labels={
                    "ip": "Host IP Address",
                    "port": "Number of Open Ports"
                },
                color_discrete_sequence=px.colors.qualitative.Plotly,
                hover_data={'port': True}
            )
            st.plotly_chart(ports_chart, use_container_width=True)

        with col2:
            risk_chart = px.bar(
                df.groupby("ip")["risk_score"].max().reset_index(), # Using max risk score per IP for this chart
                x="ip",
                y="risk_score",
                title="<b>Highest Risk Score per Host</b>", # Changed title to reflect 'max'
                labels={
                    "ip": "Host IP Address",
                    "risk_score": "Highest Risk Score"
                },
                color="risk_score",
                color_continuous_scale="Reds",
                hover_data={'risk_score': True}
            )
            st.plotly_chart(risk_chart, use_container_width=True)

        col3,col4 = st.columns(2)
        with col3:
            service_chart = px.bar(
                df["service"].value_counts().reset_index(name='count', inplace=False),
                x="count",
                y="service",
                orientation="h",
                title="<b>Services Exposed Across All Hosts</b>",
                labels={
                    "count": "Number of Occurrences",
                    "service": "Service Name"
                },
                color_discrete_sequence=px.colors.qualitative.Pastel,
                hover_data={'count': True}
            )
            st.plotly_chart(service_chart, use_container_width=True)

        with col4:
            severity_chart = px.pie(
                df,
                names="severity",
                title="<b>Severity Distribution of Findings</b>",
                hole=0.4,
                color="severity",
                color_discrete_map={'Critical':'#8B0000', 'High':'red', 'Medium':'orange', 'Low':'green'}
            )
            st.plotly_chart(severity_chart, use_container_width=True)

        st.markdown("### <b>Risk Heatmap: IP vs. Port</b>")
        # Aggregate data for heatmap to ensure unique IP-Port combinations
        heatmap_data = df.groupby(['ip', 'port'])['risk_score'].max().reset_index()

        heatmap_chart = px.density_heatmap(
            heatmap_data,
            x="ip",
            y="port",
            z="risk_score",
            title="<b>Risk Heatmap: IP vs. Port</b>",
            labels={
                "ip": "IP Address",
                "port": "Port Number",
                "risk_score": "Risk Score"
            },
            color_continuous_scale="Reds",
            height=600 # Adjust height for better visibility
        )
        st.plotly_chart(heatmap_chart, use_container_width=True)

    else:
        st.info("No scan data to display charts. Run a full scan.")

with tab3:
    st.header("\U0001F6A8 Threat Intelligence Summary")
    if not df.empty:
        st.markdown("### Shodan Insights")
        if "shodan_country" in df.columns and "shodan_vuln_count" in df.columns:
            shodan_summary = df.groupby("shodan_country").agg(num_ips=("ip","nunique"), total_vulns=("shodan_vuln_count", "sum")).reset_index()
            st.dataframe(shodan_summary, use_container_width=True)
        else:
            st.info("Shodan insights not available (missing 'shodan_country' or 'shodan_vuln_count' column).")

        high_risk = df[df["severity"].isin(["High", "Critical"])] # Include Critical for alert
        if len(high_risk)>0:
            st.error(
                f"\u26A0\ufe0f **CRITICAL ALERT:** {len(high_risk)} high-risk entries across "
                f"{high_risk['ip'].nunique()} host(s). Immediate action required."
            )
            with st.expander("\U0001F534 Show High Risk Entries Details", expanded=True):
                st.dataframe(high_risk, use_container_width=True, hide_index=True)
        else:
            st.success("\u2705 No high risk entries detected. Your network appears secure.")

        st.markdown("---")
        st.subheader("Manual Actions")
        if st.button("Send Alert Email (Based on Current Data)"):
            if not df.empty:
                email_status = send_email_alert(df, "Manual Alert Trigger")
                if "successfully" in email_status:
                    st.success(email_status)
                else:
                    st.warning(email_status)
            else:
                st.info("No scan data available to send alerts.")

    else:
        st.info("No scan data available for threat intelligence. Run a full scan.")

with tab4:
    st.header("\U0001F4BE Export Scan Results")
    if not df.empty:
        st.write("Download the full scan results as a CSV file.")
        csv = df.to_csv(index=False).encode("utf-8")
        st.download_button(
            label="Download CSV",
            data=csv,
            file_name="cyber_scan_results.csv",
            mime="text/csv",
            help="Click to download the current scan results as a CSV file."
        )

        st.write("Download a PDF report of high-risk findings.")
        if st.button("Download PDF Report", help="Generate and download a PDF report of high-risk vulnerabilities."):
            with st.spinner("Generating PDF report..."):
                pdf_file_path = generate_pdf_report(df, "Overall Scan Results")
                with open(pdf_file_path, "rb") as pdf_file:
                    st.download_button(
                        label="Click to Download PDF",
                        data=pdf_file.read(),
                        file_name="cyber_scan_report.pdf",
                        mime="application/pdf",
                        key="pdf_download_button"
                    )
                os.remove(pdf_file_path) # Clean up the generated PDF file

    else:
        st.info("No scan data to export. Please run a scan first.")

Overwriting dashboard.py


In [ ]:
import subprocess
import threading

def run_streamlit():

    subprocess.run([
        "streamlit",
        "run",
        "dashboard.py",
        "--server.port","8501",
        "--server.headless","true"
    ])

threading.Thread(target=run_streamlit).start()

In [ ]:
def check_shodan(ip):
    if not SHADON_API_KEY or SHADON_API_KEY == "G4XUuqWCIvU0kUKMkNkkKeHxp1AGQYB9":
        return {"shodan_country": "N/A", "shodan_vuln_count": 0, "shodan_ports": "N/A"}

    url = f"https://api.shodan.io/shodan/host/{ip}?key={SHADON_API_KEY}"
    try:
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            data = response.json()
            country = data.get("country_name", "Unknown")
            vuln_count = len(data.get("vulns", [])) if isinstance(data.get("vulns"), list) else 0
            if isinstance(data.get("vulns"), dict):
                vuln_count = len(data.get("vulns").keys())

            ports = data.get("ports",[])
            shodan_ports = ", ".join(map(str, ports)) if ports else "N/A"

            return {"shodan_country": country, "shodan_vuln_count": vuln_count, "shodan_ports": shodan_ports}
        else:
            return {"shodan_country": "Unknown", "shodan_vuln_count": 0, "shodan_ports": "N/A"}
    except requests.exceptions.RequestException:
        return {"shodan_country": "Error", "shodan_vuln_count": 0, "shodan_ports": "N/A"}

print("check_shodan function defined.")

check_shodan function defined.


In [ ]:
shodan_results = []
for ip in df["ip"].unique():
    shodan_data = check_shodan(ip)
    shodan_results.append({"ip": ip, **shodan_data})

df_shodan = pd.DataFrame(shodan_results)
df = pd.merge(df, df_shodan, on="ip", how="left").fillna({"shodan_country": "N/A", "shodan_vuln_count": 0, "shodan_ports": "N/A"})

print("Shodan data merged into DataFrame.")
df.head()

Shodan data merged into DataFrame.


,ip,port,service,vt_reputation,risk_score,severity,shodan_country,shodan_vuln_count,shodan_ports
0,44.238.29.244,80,http,Medium,3,Medium,N/A,0,N/A
1,54.82.22.214,80,http,Low,1,Low,N/A,0,N/A
2,54.82.22.214,443,https,Low,1,Low,N/A,0,N/A
3,54.82.22.214,8080,http,Low,1,Low,N/A,0,N/A


In [ ]:
!nohup streamlit run dashboard.py --server.port 8501 --server.headless true > streamlit.log 2>&1 &

In [ ]:
!cloudflared tunnel --url http://localhost:8501

2026-03-28T01:20:34Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-03-28T01:20:34Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-03-28T01:20:37Z INF +--------------------------------------------------------------------------------------------+
2026-03-28T01:20:37Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-03-28T01:20:37Z INF |  https://lauren-controls-toronto-reaction.trycloudflar

In [ ]:
!killall streamlit
!killall cloudflared

import subprocess
import threading

def run_streamlit():
    subprocess.run([
        "streamlit",
        "run",
        "dashboard.py",
        "--server.port","8501",
        "--server.headless","true"
    ])

threading.Thread(target=run_streamlit).start()


import subprocess

def run_cloudflared():
    subprocess.run([
        "cloudflared",
        "tunnel",
        "--url", "http://localhost:8501"
    ])

threading.Thread(target=run_cloudflared).start()

In [ ]:
import pandas as pd

# Create a dummy DataFrame with 'High' severity to test email alert
dummy_high_risk_df = pd.DataFrame({
    "ip": ["1.2.3.4"],
    "port": [22],
    "service": ["ssh"],
    "risk_score": [4],
    "vt_reputation": ["High"],
    "severity": ["High"]
})

print("Attempting to send email alert for dummy high-risk vulnerabilities...")
send_email_alert(dummy_high_risk_df, "dummy_test_target.com")
print("Verification complete.")

Attempting to send email alert for dummy high-risk vulnerabilities...
Alert email sent
Verification complete.
